In [1]:
import numpy as np

# 1. Dimension Setup from image data
num_samples = 100  # m
num_features = 200 # Dx
hidden_neurons = 20 # Da1
num_classes = 10   # C (class labels 0 to 9)

# 2. task (a): Generate random synthetic samples X in R^(m x Dx)
# We will generate X with shape (num_samples, num_features) to be standard
# for vectorization. X = [x^(1) x^(2) ... x^(m)]^T
X = np.random.randn(num_samples, num_features) # Use a normal distribution for a first pass
print(f"(a) Shape of synthetic input data X: {X.shape}")

# 3. task (b): Generate random class labels and convert to one-hot vector Y in R^(m x C)
# First, generate random class labels (0 to 9)
np.random.seed(42) # For reproducibility
integer_labels = np.random.randint(0, num_classes, num_samples)

# Then, create one-hot encoding from scratch
# Initialize a matrix of zeros
Y = np.zeros((num_samples, num_classes))
# Set '1' at the index corresponding to each sample's integer label
# This is a concise way to do it: array indexing to set elements to 1
Y[np.arange(num_samples), integer_labels] = 1

print(f"(b) Shape of one-hot encoded labels Y: {Y.shape}")
print(f"Sample integer label: {integer_labels[0]} -> One-hot vector: {Y[0]}")

(a) Shape of synthetic input data X: (100, 200)
(b) Shape of one-hot encoded labels Y: (100, 10)
Sample integer label: 6 -> One-hot vector: [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]


In [ ]:
# Function to perform a forward pass. Note: no ML libraries like tensorflow or pytorch.

def tanh(z):
    """Computes the hyperbolic tangent activation."""
    return (np.exp(z) - np.exp(-z)) / (np.exp(z) + np.exp(-z))

def softmax(z):
    """Computes the softmax activation for a batch of predictions."""
    # To handle potential numerical overflow/underflow, we use the numerically stable softmax.
    # We subtract the max value from z before computing the exponent.

    
    exps = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)

def forward_pass(X, W1, b1, W2, b2):
    """
    Computes a forward pass through the network.
    The vectorized implementation handles all samples at once.
    
    Returns the network's final output probabilities (hat_y)
    """
    # X shape: (num_samples, num_features)
    # W1 shape: (hidden_neurons, num_features) -> must be transposed
    # b1 shape: (hidden_neurons, 1) -> must be transposed and will be broadcasted
    
    # Hidden Layer 1: z1 = W1*x + b1
    # For a batch of samples X: z1 = X @ W1^T + b1^T
    z1 = np.dot(X, W1.T) + b1.T 
    # a1 = tanh(z1)
    a1 = tanh(z1)
    
    # Output Layer 2: z2 = W2*a1 + b2
    # For a batch: z2 = a1 @ W2^T + b2^T
    # W2 shape: (num_classes, hidden_neurons) -> must be transposed
    # b2 shape: (num_classes, 1) -> must be transposed and will be broadcasted
    z2 = np.dot(a1, W2.T) + b2.T
    # hat_y = softmax(z2)
    hat_y = softmax(z2)
    
    return hat_y

def compute_average_loss(hat_y, y):
    """Computes the average categorical cross-entropy loss."""
    # We add a small epsilon value to log(p) to prevent numerical errors with log(0).
    epsilon = 1e-15 
    m = y.shape[0]
    # The loss function is: L^(i) = -sum(y_j^(i) * log(hat_y_j^(i)))
    # For a vectorized matrix operation, this can be computed as:
    # J = -(1/m) * sum(elementwise y * log(hat_y))
    total_loss = -np.sum(y * np.log(hat_y + epsilon)) 
    average_loss = total_loss / m
    return average_loss

def calculate_macro_metrics(y_true_onehot, y_pred_probs, num_classes=10):
    """
    Calculates macro-averaged Accuracy, Precision, Recall, and F1-score from scratch.
    """
    m = y_true_onehot.shape[0]
    
    # 1. Convert probabilistic predictions and one-hot labels back to class indices
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    y_true_classes = np.argmax(y_true_onehot, axis=1)

    # 2. Accuracy: (Number of correct predictions) / (Total predictions)
    correct_predictions = np.sum(y_pred_classes == y_true_classes)
    accuracy = correct_predictions / m

    # 3. Precision, Recall, F1-score for each class (Macro Averaging)
    per_class_precision = []
    per_class_recall = []
    per_class_f1 = []
    
    # We use macro averaging: calculate scores per class and then average.
    # This is better for imbalanced datasets and is the most general from-scratch solution.
    for c in range(num_classes):
        # Count outcomes for class 'c'
        # True Positive (TP): Predicted c and is c
        tp = np.sum((y_pred_classes == c) & (y_true_classes == c))
        # False Positive (FP): Predicted c, but is not c
        fp = np.sum((y_pred_classes == c) & (y_true_classes != c))
        # False Negative (FN): Is c, but did not predict c
        fn = np.sum((y_pred_classes != c) & (y_true_classes == c))
        
        # Calculate scores with a check for division by zero
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        per_class_precision.append(precision)
        per_class_recall.append(recall)
        per_class_f1.append(f1)
        
    # Average the per-class scores for the final macro metrics
    macro_precision = np.mean(per_class_precision)
    macro_recall = np.mean(per_class_recall)
    macro_f1 = np.mean(per_class_f1)
    
    return accuracy, macro_precision, macro_recall, macro_f1

In [3]:
# (c) Implement the neural network from scratch
# (d) Use randomly initialized weights and biases for one forward pass

# Random Parameter Initialization
# Shapes of parameters:
# W1 is (neurons_in_layer_1, features_in) = (20, 200)
# b1 is (neurons_in_layer_1, 1) = (20, 1)
# W2 is (neurons_in_layer_out, neurons_in_layer_1) = (10, 20)
# b2 is (neurons_in_layer_out, 1) = (10, 1)

W1 = np.random.randn(hidden_neurons, num_features) * 0.01 # Small random numbers are a good start
b1 = np.random.randn(hidden_neurons, 1) * 0.01 
W2 = np.random.randn(num_classes, hidden_neurons) * 0.01
b2 = np.random.randn(num_classes, 1) * 0.01

# Execute one forward pass using the initialized weights and generated data
hat_Y_probs = forward_pass(X, W1, b1, W2, b2)
print(f"Shape of output probabilities hat_Y: {hat_Y_probs.shape}")

# (e) Compute the average cross-entropy loss
# Compute loss between the network's predictions and the true one-hot labels
average_loss = compute_average_loss(hat_Y_probs, Y)
print(f"(e) Average cross-entropy loss: {average_loss:.6f}")

Shape of output probabilities hat_Y: (100, 10)
(e) Average cross-entropy loss: 2.301748


In [4]:
# (f) Calculate and display the following performance metrics: 
#     Accuracy, Precision, Recall, F1-score

# Use the function from Cell 2 to compute all required metrics from scratch
accuracy, precision, recall, f1_score = calculate_macro_metrics(Y, hat_Y_probs, num_classes)

# Display the final performance summary
# Since weights are random, these scores are expected to be poor.
# We are only verifying the computational flow is correct.

print(f"(f) Final Performance Metrics (calculated on a random model):")
print("-" * 35)
print(f"  Accuracy:         {accuracy * 100:>6.2f}%")
# We specify "Macro" to indicate we're averaging scores across all classes.
print(f"  Macro Precision:  {precision:>6.3f}")
print(f"  Macro Recall:     {recall:>6.3f}")
print(f"  Macro F1-score:   {f1_score:>6.3f}")
print("-" * 35)

(f) Final Performance Metrics (calculated on a random model):
-----------------------------------
  Accuracy:           8.00%
  Macro Precision:   0.039
  Macro Recall:      0.072
  Macro F1-score:    0.046
-----------------------------------
